In [1]:
!pip install sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.6 MB/s eta 0:00:00


In [2]:
import os
import copy 
import random
import warnings
from pathlib import Path
from collections import Counter

import numpy as np
from PIL import Image

import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms, models
from torchsummary import summary

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score
)

from torch.optim.lr_scheduler import ReduceLROnPlateau
warnings.filterwarnings('ignore')

SEED = 42
import pandas as pd
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
from tqdm import tqdm
import sacrebleu

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

2026-04-20 12:17:56.653043: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776687476.848888      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776687476.909950      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776687477.398315      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776687477.398353      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776687477.398356      23 computation_placer.cc:177] computation placer alr

In [3]:
train_df = pd.read_csv("/kaggle/input/datasets/tridodai/dataset-voai-bana/kaggle_upload/train.csv")


In [4]:
texts = list(train_df["Bahnaric"].fillna("")) + list(train_df["Vietnamese"].fillna(""))

with open("corpus.txt", "w", encoding="utf-8") as f:
    for line in texts:
        f.write(line.strip() + "\n")

import sentencepiece as spm

spm.SentencePieceTrainer.train(
    input="corpus.txt",
    model_prefix="bpe",
    vocab_size=8000,
    model_type="bpe",   # 🔥 BPE
    character_coverage=1.0
)

sp = spm.SentencePieceProcessor()
sp.load("bpe.model")

PAD_ID = 0
BOS_ID = 1
EOS_ID = 2



sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: corpus.txt
  input_format: 
  model_prefix: bpe
  model_type: BPE
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 1
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 0
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0
  differential_privacy_

In [5]:

MAX_LEN = 100

def encode(text):
    ids = sp.encode(text, out_type=int)
    ids = [BOS_ID] + ids[:MAX_LEN-2] + [EOS_ID]
    ids += [PAD_ID] * (MAX_LEN - len(ids))
    return ids

class TranslationDataset(Dataset):
    def __init__(self, df):
        self.src = df["Bahnaric"].fillna("").tolist()
        self.tgt = df["Vietnamese"].fillna("").tolist()

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        src_ids = encode(self.src[idx])
        tgt_ids = encode(self.tgt[idx])

        return {
            "src": torch.tensor(src_ids),
            "tgt": torch.tensor(tgt_ids)
        }

In [6]:
import torch.nn as nn
import torch
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.3, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Sử dụng hàm Sin/Cos thay vì Parameter học được để chống overfit
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0) # (1, max_len, d_model) cho batch_first=True
        self.register_buffer('pe', pe)

    def forward(self, x):
        # Cộng positional encoding vào input
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class Seq2SeqTransformer(nn.Module):
    # Tăng dropout mặc định lên 0.3 vì dữ liệu dân tộc thường rất dễ overfit
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=3, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout)

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True  # <--- CỰC KỲ QUAN TRỌNG: Pre-LayerNorm
        )
        self.fc = nn.Linear(d_model, vocab_size)

        # Weight Tying: Chia sẻ trọng số giữa layer Embedding và layer Linear cuối
        self.fc.weight = self.embedding.weight

        self._reset_parameters()

    def _reset_parameters(self):
        # Khởi tạo trọng số Xavier Uniform giúp ổn định gradient những epoch đầu
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, src, tgt, tgt_mask=None, src_key_padding_mask=None, tgt_key_padding_mask=None):
        # Scale embedding theo căn bậc 2 của d_model
        src_emb = self.embedding(src) * math.sqrt(self.embedding.embedding_dim)
        tgt_emb = self.embedding(tgt) * math.sqrt(self.embedding.embedding_dim)

        # Pass qua Positional Encoding (có kèm dropout)
        src_emb = self.pos_encoder(src_emb)
        tgt_emb = self.pos_encoder(tgt_emb)

        out = self.transformer(
            src_emb, 
            tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask
        )
        return self.fc(out)

In [7]:
from sklearn.model_selection import train_test_split
train_data, val_data = train_test_split(train_df, test_size=0.1)

train_loader = DataLoader(TranslationDataset(train_data), batch_size=32, shuffle=True,num_workers=4)
val_loader = DataLoader(TranslationDataset(val_data), batch_size=32,num_workers=4)

model = Seq2SeqTransformer(vocab_size=8000).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

In [8]:
def train_epoch():
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader):
        src = batch["src"].to(device)
        tgt = batch["tgt"].to(device)

        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]

        # 1. Tạo Causal Mask (chống nhìn trộm tương lai)
        tgt_mask = model.transformer.generate_square_subsequent_mask(tgt_input.size(1)).to(device)
        
        # 2. Tạo Padding Mask (bỏ qua khoảng đệm)
        src_padding_mask = (src == PAD_ID).to(device)
        tgt_padding_mask = (tgt_input == PAD_ID).to(device)

        logits = model(
            src, 
            tgt_input, 
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_padding_mask,
            tgt_key_padding_mask=tgt_padding_mask
        )

        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            tgt_output.reshape(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

In [9]:
def greedy_decode(src):
    model.eval()

    src = src.unsqueeze(0).to(device)
    tgt = torch.tensor([[BOS_ID]], device=device)

    for _ in range(MAX_LEN):
        out = model(src, tgt)
        next_token = out[:, -1].argmax(-1).unsqueeze(1)
        tgt = torch.cat([tgt, next_token], dim=1)

        if next_token.item() == EOS_ID:
            break

    return tgt.squeeze().tolist()

def evaluate():
    model.eval()
    preds, refs = [], []

    with torch.no_grad():
        for batch in tqdm(val_loader):
            src = batch["src"]
            tgt = batch["tgt"]

            for i in range(len(src)):
                pred_ids = greedy_decode(src[i])
                pred_text = sp.decode(pred_ids)

                ref_text = sp.decode(tgt[i].tolist())

                preds.append(pred_text)
                refs.append(ref_text)

    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    return bleu

In [10]:
EPOCHS = 5

for epoch in range(EPOCHS):
    train_loss = train_epoch()
    val_bleu = evaluate()

    print(f"Epoch {epoch}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Val BLEU: {val_bleu:.2f}")

100%|██████████| 130/130 [14:03<00:00,  6.48s/it]


Epoch 0
Train Loss: 5.4696
Val BLEU: 0.04


100%|██████████| 130/130 [07:49<00:00,  3.61s/it]


Epoch 1
Train Loss: 4.6828
Val BLEU: 0.07


100%|██████████| 130/130 [07:54<00:00,  3.65s/it]


Epoch 2
Train Loss: 4.3605
Val BLEU: 0.14


100%|██████████| 130/130 [06:59<00:00,  3.23s/it]


Epoch 3
Train Loss: 4.1460
Val BLEU: 0.11


100%|██████████| 130/130 [07:33<00:00,  3.49s/it]


Epoch 4
Train Loss: 3.9886
Val BLEU: 0.15


In [11]:
test_df = pd.read_csv("/kaggle/input/datasets/tridodai/dataset-voai-bana/kaggle_upload/test.csv")
sample_sub = pd.read_csv("/kaggle/input/datasets/tridodai/dataset-voai-bana/kaggle_upload/sample_submission.csv")

predictions = []

for text in tqdm(test_df["Bahnaric"].fillna("")):
    src_ids = torch.tensor(encode(text))
    pred_ids = greedy_decode(src_ids)
    pred_text = sp.decode(pred_ids)

    predictions.append(pred_text)

id_col = sample_sub.columns[0]
target_col = [c for c in sample_sub.columns if c != id_col][0]

sub = sample_sub.copy()
sub[target_col] = predictions
sub.to_csv("submission.csv", index=False, encoding="utf-8-sig")

print("Done!")

100%|██████████| 10386/10386 [32:16<00:00,  5.36it/s]


Done!
